# M16 · 08 — GGUF export and installation in the app

`python -m llm.export_gguf` writes an f16 GGUF (architecture `llama`, our BPE tokenizer with the
`llama-bpe` pre-tokenizer, ChatML template, EOS `<|im_end|>`). It goes into
`backend/models/llm/<label>/` with a `manifest.json` (`file`, `n_ctx`, `label`); the app reads
`LLM_OWN_DIR` (default `models/llm/current`, a symlink to the chosen export). Qwen stays the
global default; our model is selectable in Settings and per tool (0025 §11 and amendment).

In [1]:
import json, statistics
from pathlib import Path
from llm import HOME
RUNS, SFT, EVAL = HOME / 'runs', HOME / 'data' / 'sft', HOME / 'eval'
def load(p): return json.loads(Path(p).read_text())
def metrics(run): return [json.loads(l) for l in (RUNS / run / 'metrics.jsonl').read_text().splitlines()]
APP = Path('~/projects/new_youtube_suite/backend/models/llm').expanduser()
for d in sorted(APP.iterdir()):
    if (d / 'manifest.json').exists():
        m = load(d / 'manifest.json'); f = d / m['file']
        print(f"{d.name:22s} {'-> ' + str(d.resolve().name) if d.is_symlink() else '':24s} {f.stat().st_size/1e6:7.1f} MB  {m}")

current                -> yts-own-110m-dpo        202.6 MB  {'file': 'yts-own-110m-dpo.gguf', 'n_ctx': 4096, 'label': 'yts-own-110m-dpo', 'note': 'M16 stage A: 110M from scratch, 5B tokens pretraining + general SFT + task SFT + task DPO (a3-dpo)'}
yts-own-110m-dpo                                  202.6 MB  {'file': 'yts-own-110m-dpo.gguf', 'n_ctx': 4096, 'label': 'yts-own-110m-dpo', 'note': 'M16 stage A: 110M from scratch, 5B tokens pretraining + general SFT + task SFT + task DPO (a3-dpo)'}
yts-own-110m-sft                                  202.6 MB  {'file': 'yts-own-110m-sft.gguf', 'n_ctx': 4096, 'label': 'yts-own-110m-sft', 'note': 'M16 stage A: 110M from scratch, 5B tokens pretraining + general SFT + task SFT (a3-sft-tasks-e1)'}


### Parity: PyTorch vs llama.cpp on the same prompt

In [2]:
import torch, os, sys
from llm import config as cfgmod
from llm.model import Decoder
from llm.tokenizer import load_tokenizer
from llm.export_gguf import preload_llama_cpp
cur = APP / 'current'; man = load(cur / 'manifest.json')
run = {'yts-own-110m-sft': 'a3-sft-tasks-e1', 'yts-own-110m-dpo': 'a3-dpo'}[man['label']]
ck = torch.load(RUNS / run / 'final_model.pt', map_location='cpu', weights_only=False)
model = Decoder(cfgmod.from_dict(ck['config']).model); model.load_state_dict({k.removeprefix('_orig_mod.'): v for k, v in ck['model'].items()}); model.eval()
tok = load_tokenizer(HOME / 'tokenizer' / 'tokenizer.json')
prompt = '<|im_start|>user\nResume en una frase: el episodio habla de cómo empezar un podcast.<|im_end|>\n<|im_start|>assistant\n'
ids = tok.encode(prompt).ids
with torch.no_grad(): ref = model(torch.tensor([ids]))[0, -1].float()
preload_llama_cpp()
from llama_cpp import Llama
llm = Llama(model_path=str(cur / man['file']), n_ctx=512, logits_all=True, verbose=False, n_gpu_layers=0)
llm.eval(ids); got = torch.tensor(llm.scores[len(ids) - 1])
print('top-5 torch    :', ref.topk(5).indices.tolist()); print('top-5 llama.cpp:', got.topk(5).indices.tolist())
print('max |logit diff|:', float((ref - got).abs().max()))

top-5 torch    : [654, 16, 1065, 801, 36]
top-5 llama.cpp: [654, 16, 1065, 801, 36]
max |logit diff|: 0.015295982360839844
